In [16]:
import pandas as pd
import numpy as np
import sqlite3
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [17]:
conn = sqlite3.connect("employee.db")
df = pd.read_csv("employees_cleaned.csv")
df.to_sql("employees", conn, if_exists="replace", index=False)
df = pd.read_sql("SELECT * FROM employees", conn)
print(df.info())
print("Employee Data Loaded.")

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  30 non-null     int64
 1   EmployeeID  30 non-null     int64
 2   FirstName   30 non-null     str  
 3   LastName    30 non-null     str  
 4   Department  30 non-null     str  
 5   DOB         30 non-null     str  
 6   Email       30 non-null     str  
 7   Phone       24 non-null     str  
 8   Salary      30 non-null     int64
 9   City        30 non-null     str  
dtypes: int64(3), str(7)
memory usage: 2.5 KB
None
Employee Data Loaded.


In [20]:
df["DOB"] = pd.to_datetime(df["DOB"])
df["Age"] = 2026 - df["DOB"].dt.year
df = df.drop(["EmployeeID", "FirstName", "LastName", "Email", "Phone", "DOB"], axis=1)
le = LabelEncoder()
df["City"] = le.fit_transform(df["City"])
df["Department"] = le.fit_transform(df["Department"])

In [21]:
X = df.drop("Department", axis=1)
y = df["Department"]

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
model = RandomForestClassifier(n_estimators=200, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
pickle.dump(model, open("employee_model.pkl", "wb"))
pickle.dump(scaler, open("employee_scaler.pkl", "wb"))

print("Employee Model Saved.")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      0.67      0.80         3
           3       0.50      1.00      0.67         1

    accuracy                           0.83         6
   macro avg       0.83      0.89      0.82         6
weighted avg       0.92      0.83      0.84         6

Employee Model Saved.
